# CoreGripper + Arm: Simple Resource Move Test

Tests the `CoreGripper` backend through `Arm` with real PLR resources and the real STAR firmware interface.

Tool management (pick up / return gripper tools) is still handled by the STAR backend.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pylabrobot.arms.arm import Arm
from pylabrobot.hamilton.liquid_handlers.star.core import CoreGripper
from pylabrobot.legacy.liquid_handling.backends.hamilton.STAR_backend import STARBackend
from pylabrobot.resources.hamilton.hamilton_decks import STARDeck
from pylabrobot.resources.hamilton.plate_carriers import PLT_CAR_L5AC_A00
from pylabrobot.resources.corning.plates import Cor_96_wellplate_360ul_Fb

## Set up deck with carrier and plate

In [3]:
deck = STARDeck(core_grippers="1000uL-at-waste")

carrier = PLT_CAR_L5AC_A00("carrier")
deck.assign_child_resource(carrier, rails=14)

plate = Cor_96_wellplate_360ul_Fb("my_plate")
carrier[1].assign_child_resource(plate)

print(f"Plate '{plate.name}' is in: {plate.parent.name}")
print(f"Plate absolute location: {plate.get_absolute_location()}")
print(f"Destination site: {carrier[2].name}")
print(f"Destination location: {carrier[2].get_absolute_location()}")

Plate 'my_plate' is in: carrier-1
Plate absolute location: Coordinate(396.500, 167.500, 183.120)
Destination site: carrier-2
Destination location: Coordinate(396.500, 263.500, 186.150)


## Create CoreGripper backend with real STAR interface

In [4]:
star_backend = STARBackend()
star_backend.set_deck(deck)
await star_backend.setup()

core_backend = CoreGripper(interface=star_backend)

arm = Arm(backend=core_backend, reference_resource=deck, grip_axis="y")
print("Arm ready")

2026-03-21 15:31:31,715 - pylabrobot.io.usb - INFO - Finding USB device...
2026-03-21 15:31:31,720 - pylabrobot.io.usb - INFO - Found USB device.
2026-03-21 15:31:31,720 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0


Arm ready


## Pick up core gripper tools

Tool management is handled by the legacy STAR backend for now.

In [5]:
await star_backend.pick_up_core_gripper_tools(front_channel=7)
print(f"Core gripper tools picked up, core_parked: {star_backend.core_parked}")

Core gripper tools picked up, core_parked: False


## Pick up plate from carrier[1]

In [6]:
arm._end_holding()

In [7]:
await arm.pick_up_resource(plate)
print(f"Picked up '{plate.name}'")

Picked up 'my_plate'


## Drop plate at carrier[2]

In [8]:
await arm.drop_resource(carrier[0])
print(f"Plate '{plate.name}' is now in: {plate.parent.name}")
print(f"Plate absolute location: {plate.get_absolute_location()}")

Plate 'my_plate' is now in: carrier-0
Plate absolute location: Coordinate(396.500, 071.500, 183.120)


## Return core gripper tools

In [ ]:
await star_backend.return_core_gripper_tools()
print(f"Core gripper tools returned, core_parked: {star_backend.core_parked}")